# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 7.4 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile, glob, sys,math, random, collections, csv
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict


In [5]:
TASK_ID = "task037"
CH = 10
H = W = 30
CANVAS = 10
MAX_ONNX_BYTES = 1_400_000
FORBIDDEN_OPS = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
ONNX_PATH = WORKDIR / f"{TASK_ID}_static_graph.onnx"
ZIP_PATH = WORKDIR / f"{TASK_ID}_static_graph_submission.zip"
GENERIC_ZIP = WORKDIR / "submission.zip"
HEALTH_PATH = WORKDIR / f"{TASK_ID}_verified_onnx_health.json"

In [6]:
def find_task_json(task_id: str) -> Path:
    candidates = [
        Path.cwd() / f"{task_id}.json",
        Path("/mnt/data") / f"{task_id}.json",
        Path("/kaggle/working") / f"{task_id}.json",
    ]
    for p in candidates:
        if p.exists():
            return p
    for root in [Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            matches = list(root.rglob(f"{task_id}.json"))
            if matches:
                return matches[0]
    raise FileNotFoundError(f"Could not find {task_id}.json")

TASK_PATH = find_task_json(TASK_ID)
with open(TASK_PATH) as f:
    task = json.load(f)
print("Loaded", TASK_PATH)
print({k: len(task.get(k, [])) for k in ["train", "test", "arc-gen"]})

Loaded /kaggle/input/competitions/neurogolf-2026/task037.json
{'train': 3, 'test': 1, 'arc-gen': 262}


In [7]:
def encode_grid(grid, H: int = 30, W: int = 30) -> np.ndarray:
    arr = np.array(grid, dtype=np.int64)
    x = np.zeros((1, 10, H, W), dtype=np.float32)
    h, w = arr.shape
    assert h <= H and w <= W
    for c in range(10):
        x[0, c, :h, :w] = (arr == c).astype(np.float32)
    return x


def decode_onehot(y: np.ndarray, h: int, w: int) -> np.ndarray:
    if y.ndim == 4:
        y = y[0]
    return y[:, :h, :w].argmax(axis=0).astype(np.int64)


def grid_shape(ex):
    return len(ex["input"]), len(ex["input"][0])

In [8]:
def build_diagonal_pair_tensors(n: int = 10):
    pairs = []
    paths = []
    for r1 in range(n):
        for c1 in range(n):
            for r2 in range(n):
                for c2 in range(n):
                    if (r2, c2) <= (r1, c1):
                        continue
                    dr = r2 - r1
                    dc = c2 - c1
                    if dr != 0 and abs(dr) == abs(dc):
                        steps = abs(dr)
                        sr = 1 if dr > 0 else -1
                        sc = 1 if dc > 0 else -1
                        path = [(r1 + k * sr, c1 + k * sc) for k in range(steps + 1)]
                        pairs.append((r1 * n + c1, r2 * n + c2))
                        paths.append([rr * n + cc for rr, cc in path])

    p = len(pairs)
    a = torch.zeros(p, n * n, dtype=torch.float32)
    b = torch.zeros(p, n * n, dtype=torch.float32)
    m = torch.zeros(p, n * n, dtype=torch.float32)
    for i, ((u, v), path) in enumerate(zip(pairs, paths)):
        a[i, u] = 1.0
        b[i, v] = 1.0
        for q in path:
            m[i, q] = 1.0
    return a, b, m


class Task037DiagonalConnectModel(nn.Module):
    def __init__(self):
        super().__init__()
        a, b, m = build_diagonal_pair_tensors(CANVAS)
        self.register_buffer("pair_a", a)
        self.register_buffer("pair_b", b)
        self.register_buffer("path_mask", m)

    def forward(self, x):
        patch = x[:, :, :CANVAS, :CANVAS]
        seeds = patch[:, 1:, :, :].reshape(1, 9, CANVAS * CANVAS)

        has_a = torch.matmul(seeds, self.pair_a.t())
        has_b = torch.matmul(seeds, self.pair_b.t())
        active_pair = has_a * has_b
        draw = (torch.matmul(active_pair, self.path_mask) > 0.5).to(x.dtype)
        draw = draw.reshape(1, 9, CANVAS, CANVAS)

        bg = patch[:, 0:1, :, :]
        existing_non_bg = patch[:, 1:, :, :]
        any_draw = (draw.sum(dim=1, keepdim=True) > 0.5).to(x.dtype)
        y0 = bg * (1.0 - any_draw)
        ycols = existing_non_bg + draw * bg
        ypatch = torch.cat([y0, ycols], dim=1)

        top_rows = torch.cat([ypatch, x[:, :, :CANVAS, CANVAS:]], dim=3)
        return torch.cat([top_rows, x[:, :, CANVAS:, :]], dim=2)


model = Task037DiagonalConnectModel().eval()

In [9]:
def run_torch(ex):
    h, w = grid_shape(ex)
    x = torch.tensor(encode_grid(ex["input"]), dtype=torch.float32)
    with torch.no_grad():
        y = model(x).cpu().numpy()
    return decode_onehot(y, h, w)

for split in ["train", "test", "arc-gen"]:
    ok = 0
    bad = []
    for i, ex in enumerate(task.get(split, [])):
        pred = run_torch(ex)
        gold = np.array(ex["output"], dtype=np.int64)
        good = np.array_equal(pred, gold)
        ok += int(good)
        if not good and len(bad) < 5:
            bad.append(i)
    print(split, ok, "/", len(task.get(split, [])), "bad", bad)
    assert ok == len(task.get(split, []))

train 3 / 3 bad []
test 1 / 1 bad []
arc-gen 262 / 262 bad []


In [10]:
dummy = torch.zeros(1, CH, H, W, dtype=torch.float32)
dummy[:, 0, :CANVAS, :CANVAS] = 1.0
dummy[:, 0, 0, 0] = 0.0
dummy[:, 4, 0, 0] = 1.0
dummy[:, 0, 3, 3] = 0.0
dummy[:, 4, 3, 3] = 1.0

try:
    torch.onnx.export(
        model,
        dummy,
        str(ONNX_PATH),
        input_names=["input"],
        output_names=["output"],
        opset_version=17,
        do_constant_folding=True,
        dynamic_axes=None,
        dynamo=False,
    )
except TypeError:
    torch.onnx.export(
        model,
        dummy,
        str(ONNX_PATH),
        input_names=["input"],
        output_names=["output"],
        opset_version=17,
        do_constant_folding=True,
        dynamic_axes=None,
    )
print("Wrote", ONNX_PATH, "size", ONNX_PATH.stat().st_size)

/tmp/ipykernel_16/3842593799.py:9: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Wrote /kaggle/working/task037_static_graph.onnx size 688771


In [11]:
onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
ops = sorted({node.op_type for node in onnx_model.graph.node})
forbidden_found = sorted(FORBIDDEN_OPS.intersection(ops))
input_shape = [d.dim_value for d in onnx_model.graph.input[0].type.tensor_type.shape.dim]
output_shape = [d.dim_value for d in onnx_model.graph.output[0].type.tensor_type.shape.dim]
size_bytes = ONNX_PATH.stat().st_size

print("input_shape", input_shape)
print("output_shape", output_shape)
print("size_bytes", size_bytes)
print("forbidden_found", forbidden_found)
print("ops", ops)

assert input_shape == [1, 10, 30, 30]
assert output_shape == [1, 10, 30, 30]
assert size_bytes < MAX_ONNX_BYTES
assert not forbidden_found

input_shape [1, 10, 30, 30]
output_shape [1, 10, 30, 30]
size_bytes 688771
forbidden_found []
ops ['Add', 'Cast', 'Concat', 'Constant', 'Greater', 'MatMul', 'Mul', 'ReduceSum', 'Reshape', 'Slice', 'Sub']


In [12]:
sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])

def run_onnx(ex):
    h, w = grid_shape(ex)
    y = sess.run(None, {"input": encode_grid(ex["input"])})[0]
    return decode_onehot(y, h, w)

summary = {}
for split in ["train", "test", "arc-gen"]:
    ok = 0
    bad = []
    for i, ex in enumerate(task.get(split, [])):
        pred = run_onnx(ex)
        gold = np.array(ex["output"], dtype=np.int64)
        good = np.array_equal(pred, gold)
        ok += int(good)
        if not good and len(bad) < 5:
            bad.append(i)
    summary[split] = {"ok": ok, "total": len(task.get(split, [])), "bad_examples": bad}
    print(split, summary[split])

arc_gen = task.get("arc-gen", [])
cut = int(len(arc_gen) * 0.40)
for name, subset in [("arc_gen_fit_40_percent", arc_gen[:cut]), ("arc_gen_holdout_60_percent", arc_gen[cut:])]:
    ok = 0
    bad = []
    for i, ex in enumerate(subset):
        pred = run_onnx(ex)
        gold = np.array(ex["output"], dtype=np.int64)
        good = np.array_equal(pred, gold)
        ok += int(good)
        if not good and len(bad) < 5:
            bad.append(i)
    summary[name] = {"ok": ok, "total": len(subset), "bad_examples": bad}
    print(name, summary[name])

assert summary["train"]["ok"] == summary["train"]["total"]
assert summary["test"]["ok"] == summary["test"]["total"]
assert summary["arc_gen_fit_40_percent"]["ok"] == summary["arc_gen_fit_40_percent"]["total"]
assert summary["arc_gen_holdout_60_percent"]["ok"] == summary["arc_gen_holdout_60_percent"]["total"]

train {'ok': 3, 'total': 3, 'bad_examples': []}
test {'ok': 1, 'total': 1, 'bad_examples': []}
arc-gen {'ok': 262, 'total': 262, 'bad_examples': []}
arc_gen_fit_40_percent {'ok': 104, 'total': 104, 'bad_examples': []}
arc_gen_holdout_60_percent {'ok': 158, 'total': 158, 'bad_examples': []}


In [13]:
health = {
    "task_id": TASK_ID,
    "model_type": "static neural-symbolic PyTorch tensor model exported to ONNX",
    "not_lookup": True,
    "uses_tree_based_method": False,
    "exact_input_output_bank": False,
    "visible_train": summary["train"],
    "visible_test": summary["test"],
    "arc_gen_all": summary["arc-gen"],
    "arc_gen_fit_40_percent": summary["arc_gen_fit_40_percent"],
    "arc_gen_holdout_60_percent": summary["arc_gen_holdout_60_percent"],
    "onnx_size_bytes": size_bytes,
    "onnx_input_shape": input_shape,
    "onnx_output_shape": output_shape,
    "forbidden_ops_found": forbidden_found,
    "onnx_ops": ops,
}
health["passes_required_checks"] = bool(
    summary["train"]["ok"] == summary["train"]["total"]
    and summary["test"]["ok"] == summary["test"]["total"]
    and summary["arc_gen_fit_40_percent"]["ok"] == summary["arc_gen_fit_40_percent"]["total"]
    and summary["arc_gen_holdout_60_percent"]["ok"] == summary["arc_gen_holdout_60_percent"]["total"]
    and size_bytes < MAX_ONNX_BYTES
    and not forbidden_found
    and input_shape == [1, 10, 30, 30]
    and output_shape == [1, 10, 30, 30]
)
with open(HEALTH_PATH, "w") as f:
    json.dump(health, f, indent=2)

for zip_path in [ZIP_PATH, GENERIC_ZIP]:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")
print("Wrote", ZIP_PATH)
print("Wrote", GENERIC_ZIP)
print(json.dumps(health, indent=2))
assert health["passes_required_checks"]

Wrote /kaggle/working/task037_static_graph_submission.zip
Wrote /kaggle/working/submission.zip
{
  "task_id": "task037",
  "model_type": "static neural-symbolic PyTorch tensor model exported to ONNX",
  "not_lookup": true,
  "uses_tree_based_method": false,
  "exact_input_output_bank": false,
  "visible_train": {
    "ok": 3,
    "total": 3,
    "bad_examples": []
  },
  "visible_test": {
    "ok": 1,
    "total": 1,
    "bad_examples": []
  },
  "arc_gen_all": {
    "ok": 262,
    "total": 262,
    "bad_examples": []
  },
  "arc_gen_fit_40_percent": {
    "ok": 104,
    "total": 104,
    "bad_examples": []
  },
  "arc_gen_holdout_60_percent": {
    "ok": 158,
    "total": 158,
    "bad_examples": []
  },
  "onnx_size_bytes": 688771,
  "onnx_input_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_output_shape": [
    1,
    10,
    30,
    30
  ],
  "forbidden_ops_found": [],
  "onnx_ops": [
    "Add",
    "Cast",
    "Concat",
    "Constant",
    "Greater",
    "MatMul",
    "Mul",